# DreamerV3 Gap-Closure Notebook

Goal: use the saved BO optimum as an expert anchor, measure the remaining BO-to-Dreamer gap, and continue Dreamer from a BO-seeded replay branch instead of launching another blind single-step run.

Working hypothesis: the remaining gap is mostly an exploration and replay-quality issue. The saved Dreamer run already reaches the BO neighborhood, so the practical intervention is to inject BO-like expert episodes into replay and continue from the current checkpoint.

In [ ]:
from __future__ import annotations

import importlib
import shutil
import uuid
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import dreamerv3_haberbosch_helpers as dreamer_helpers

hb = importlib.reload(dreamer_helpers)
hb.set_plot_style()

WORKSPACE_DIR = hb.WORKSPACE_DIR
FULL_LOGDIR = WORKSPACE_DIR / "runs" / "dreamerv3_haber_bosch"
GAP_LOGDIR = WORKSPACE_DIR / "runs" / "dreamerv3_haber_bosch_bo_seeded"
SEED = 7
RNG = np.random.default_rng(SEED)

In [ ]:
BO_REFERENCE = {
    "reward": 2712.651793,
    "normalized_action": np.array([
        -0.6127335418989747,
        0.08136597881754437,
        0.6577094487733803,
        1.0,
        1.0,
        1.0,
        -1.0,
    ], dtype=np.float32),
}

def evaluate_normalized_action(environment, action: np.ndarray) -> dict[str, float]:
    environment.reset()
    _, reward, terminated, truncated, info = environment.step(action)
    row = {
        "reward": float(reward),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
    }
    row.update({f"a{index}": float(action[index - 1]) for index in range(1, 8)})
    scaled = np.asarray(info.get("scaled_action", np.zeros_like(action)), dtype=np.float64)
    row.update({f"a{index}_physical": float(scaled[index - 1]) for index in range(1, 8)})
    for name, value in info.items():
        if name == "scaled_action":
            continue
        if isinstance(value, (int, float, np.integer, np.floating, bool)):
            row[name] = float(value) if not isinstance(value, bool) else bool(value)
    return row

preflight = hb.build_preflight_frame()
display(preflight)
if not preflight["ok"].all():
    missing = preflight.loc[~preflight["ok"], "item"].tolist()
    raise RuntimeError(f"Preflight failed for: {missing}")

env = hb.make_haber_bosch_env()
try:
    bo_reference_row = evaluate_normalized_action(env, BO_REFERENCE["normalized_action"])
finally:
    env.close()

bo_reference_frame = pd.DataFrame([bo_reference_row])
bo_reference_frame["saved_bo_reward"] = BO_REFERENCE["reward"]
bo_reference_frame["reward_replay_error"] = bo_reference_frame["saved_bo_reward"] - bo_reference_frame["reward"]
display(
    bo_reference_frame[[
        "reward",
        "saved_bo_reward",
        "reward_replay_error",
        "nh3_kg_per_s",
        "power_kw",
        "revenue_per_h",
        "energy_cost_per_h",
    ]]
)

## Why This Is A Narrow Intervention

The saved BO optimum sits on an economically strong corner of the search space. The saved Dreamer run already gets very close to it, which means the remaining problem is not that Dreamer cannot represent a good solution. The more plausible problem is that Dreamer spends too much of its single-step budget on catastrophic exploration before replay becomes useful.

That makes BO-seeded replay the cheapest testable intervention: it uses an already verified operating point, preserves the current Dreamer checkpoint, and directly attacks the startup data problem.

In [ ]:
metrics_df = hb.load_metrics_frame(FULL_LOGDIR)
train_df = metrics_df.loc[metrics_df["train_return"].notna()].copy()
train_df = train_df.sort_values("step").reset_index(drop=True)

dreamer_best_position = int(train_df["train_return"].idxmax())
dreamer_best_row = train_df.loc[dreamer_best_position].copy()
dreamer_best_reward = float(dreamer_best_row["train_return"])
dreamer_best_action = dreamer_best_row[[f"log_a{i}_normalized" for i in range(1, 8)]].to_numpy(dtype=np.float32)

headline = pd.DataFrame([
    {
        "saved_bo_reward": BO_REFERENCE["reward"],
        "best_dreamer_reward": dreamer_best_reward,
        "remaining_reward_gap": BO_REFERENCE["reward"] - dreamer_best_reward,
        "best_dreamer_step": int(dreamer_best_row["step"]),
        "best_dreamer_episode": int(dreamer_best_row["train_episode_index"]),
    }
])

comparison_rows = []
for index, name in enumerate(hb.PHYSICAL_PARAMETER_NAMES, start=1):
    bo_value = float(BO_REFERENCE["normalized_action"][index - 1])
    dreamer_value = float(dreamer_best_row[f"log_a{index}_normalized"])
    comparison_rows.append(
        {
            "parameter": name,
            "bo_normalized": bo_value,
            "dreamer_best_normalized": dreamer_value,
            "absolute_gap": abs(bo_value - dreamer_value),
        }
    )
comparison_df = pd.DataFrame(comparison_rows)

display(headline)
display(comparison_df)

In [ ]:
top_k = min(50, len(train_df))
top_train = train_df.nlargest(top_k, "train_return").copy()
positions = np.arange(1, 8)
top_median = np.array([float(top_train[f"log_a{i}_normalized"].median()) for i in range(1, 8)])
top_q25 = np.array([float(top_train[f"log_a{i}_normalized"].quantile(0.25)) for i in range(1, 8)])
top_q75 = np.array([float(top_train[f"log_a{i}_normalized"].quantile(0.75)) for i in range(1, 8)])

action_summary = pd.DataFrame(
    {
        "parameter": hb.PHYSICAL_PARAMETER_NAMES,
        "bo_normalized": BO_REFERENCE["normalized_action"],
        "dreamer_top50_median": top_median,
        "dreamer_top50_q25": top_q25,
        "dreamer_top50_q75": top_q75,
    }
)
display(action_summary)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(positions, BO_REFERENCE["normalized_action"], marker="o", linewidth=2.4, label="BO optimum")
ax.plot(positions, top_median, marker="s", linewidth=2.0, label="Dreamer top-50 median")
ax.fill_between(positions, top_q25, top_q75, alpha=0.2, label="Dreamer top-50 IQR")
ax.axhline(1.0, color="#9CA3AF", linestyle="--", linewidth=1.0)
ax.axhline(-1.0, color="#9CA3AF", linestyle="--", linewidth=1.0)
ax.set_xticks(positions)
ax.set_xticklabels([f"a{i}" for i in positions])
ax.set_ylim(-1.05, 1.05)
ax.set_xlabel("Normalized action component")
ax.set_ylabel("Value")
ax.set_title("High-Reward Action Pattern: BO vs Dreamer")
ax.legend(frameon=True)
plt.show()

## Gap-Closing Intervention

This notebook does not change Dreamer internals. It uses the cheaper intervention that the current codebase already supports:

1. Branch the current Dreamer run so the latest checkpoint, replay, and metrics are preserved.
2. Evaluate a small BO-centered action neighborhood through the one-step environment.
3. Save the best of those evaluations as Dreamer-compatible `.npz` episodes inside the branched `train_eps` directory.
4. Continue training from the existing checkpoint on the BO-seeded branch.

In [ ]:
GAP_CONTINUATION_EPISODES = 400
BO_NEIGHBORHOOD_CANDIDATES = 24
TOP_SEED_EPISODES = 12

GAP_CFG = hb.DreamerExperimentConfig()
GAP_CFG.logdir = GAP_LOGDIR
GAP_CFG.seed_logdir_from = FULL_LOGDIR
GAP_CFG.resume_from_checkpoint = "latest"
GAP_CFG.prefill_steps = 0
GAP_CFG.target_episodes = int(train_df["train_episode_index"].max()) + GAP_CONTINUATION_EPISODES
GAP_CFG.eval_every_steps = 100
GAP_CFG.log_every_steps = 25
GAP_CFG.episode_report_frequency = 10
GAP_CFG.reset_existing_run = False

gap_config_view = pd.DataFrame([
    {
        "source_logdir": str(FULL_LOGDIR),
        "branch_logdir": str(GAP_CFG.logdir),
        "resume_from_checkpoint": GAP_CFG.resume_from_checkpoint,
        "target_episodes": GAP_CFG.target_episodes,
        "prefill_steps": GAP_CFG.prefill_steps,
        "continuation_budget": GAP_CONTINUATION_EPISODES,
        "candidate_actions": BO_NEIGHBORHOOD_CANDIDATES,
        "top_seed_episodes": TOP_SEED_EPISODES,
    }
])
display(gap_config_view)

def prepare_gap_branch(config, *, reset_branch: bool = False) -> Path:
    if reset_branch and config.logdir.exists():
        archive_path = hb.archive_existing_dreamer_run(config, suffix="gap-closure-reset")
        if archive_path is not None:
            print(f"Archived existing branch to {archive_path}")
        shutil.rmtree(config.logdir)
    if not config.logdir.exists():
        if config.seed_logdir_from is not None:
            shutil.copytree(config.seed_logdir_from, config.logdir)
        else:
            config.logdir.mkdir(parents=True, exist_ok=True)
    for subdir in ("train_eps", "eval_eps", config.tensorboard_subdir, config.checkpoint_subdir):
        (config.logdir / subdir).mkdir(parents=True, exist_ok=True)
    return config.logdir

def build_single_step_episode(adapter, action: np.ndarray) -> dict[str, np.ndarray]:
    first_obs = adapter.reset()
    second_obs, reward, _, info = adapter.step(np.asarray(action, dtype=np.float32))
    action = np.asarray(action, dtype=np.float32)
    episode = {}
    for key in first_obs:
        episode[key] = np.stack([np.asarray(first_obs[key]), np.asarray(second_obs[key])], axis=0)
    episode["action"] = np.stack([np.zeros_like(action), action], axis=0).astype(np.float32)
    episode["reward"] = np.asarray([0.0, reward], dtype=np.float32)
    episode["discount"] = np.asarray([1.0, float(info.get("discount", 0.0))], dtype=np.float32)
    return episode

In [ ]:
def propose_gap_actions(bo_action: np.ndarray, dreamer_action: np.ndarray, count: int, rng: np.random.Generator) -> np.ndarray:
    proposals = [bo_action, dreamer_action]
    for alpha in np.linspace(0.0, 1.0, 9):
        proposals.append(np.clip((1.0 - alpha) * bo_action + alpha * dreamer_action, -1.0, 1.0))
    while len(proposals) < count + 10:
        anchor = bo_action if rng.random() < 0.75 else dreamer_action
        scale = 0.05 if rng.random() < 0.5 else 0.10
        proposal = np.clip(anchor + rng.normal(0.0, scale, size=bo_action.shape), -1.0, 1.0)
        proposals.append(proposal)
    deduped = []
    seen = set()
    for proposal in proposals:
        key = tuple(np.round(proposal, 6))
        if key not in seen:
            seen.add(key)
            deduped.append(np.asarray(proposal, dtype=np.float32))
    return np.asarray(deduped[: max(count, TOP_SEED_EPISODES)], dtype=np.float32)

def evaluate_seed_candidates(actions: np.ndarray) -> tuple[pd.DataFrame, dict[str, dict[str, np.ndarray]]]:
    adapter = hb.HaberBoschDreamerAdapter(seed=SEED, image_size=tuple(GAP_CFG.dummy_image_size))
    rows = []
    episodes = {}
    try:
        for index, action in enumerate(actions):
            episode = build_single_step_episode(adapter, action)
            key = f"bo-seed-{index:03d}-{uuid.uuid4().hex[:8]}"
            episodes[key] = episode
            rows.append(
                {
                    "episode_key": key,
                    "reward": float(episode["reward"][1]),
                    "nh3_kg_per_s": float(episode["log_nh3_kg_per_s"][1]),
                    "power_kw": float(episode["log_power_kw"][1]),
                    "revenue_per_h": float(episode["log_revenue_per_h"][1]),
                    "energy_cost_per_h": float(episode["log_energy_cost_per_h"][1]),
                    **{f"a{i}": float(action[i - 1]) for i in range(1, 8)},
                }
            )
    finally:
        adapter.close()
    frame = pd.DataFrame(rows).sort_values(["reward", "nh3_kg_per_s"], ascending=[False, False]).reset_index(drop=True)
    frame["rank"] = np.arange(1, len(frame) + 1)
    return frame, episodes

def write_seed_episodes(logdir: Path, ranked_frame: pd.DataFrame, episodes: dict[str, dict[str, np.ndarray]], top_k: int) -> list[str]:
    train_eps_dir = Path(logdir) / "train_eps"
    train_eps_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for row in ranked_frame.head(top_k).itertuples(index=False):
        file_path = train_eps_dir / f"{row.episode_key}-2.npz"
        np.savez_compressed(file_path, **episodes[row.episode_key])
        written.append(file_path.name)
    return written

candidate_actions = propose_gap_actions(BO_REFERENCE["normalized_action"], dreamer_best_action, BO_NEIGHBORHOOD_CANDIDATES, RNG)
candidate_actions.shape

In [ ]:
PREPARE_GAP_BRANCH = True
RESET_GAP_BRANCH = False

if PREPARE_GAP_BRANCH:
    prepare_gap_branch(GAP_CFG, reset_branch=RESET_GAP_BRANCH)
    seed_frame, seed_episodes = evaluate_seed_candidates(candidate_actions)
    written_files = write_seed_episodes(GAP_CFG.logdir, seed_frame, seed_episodes, TOP_SEED_EPISODES)
    display(seed_frame.head(TOP_SEED_EPISODES))
    display(Markdown(
        f"Prepared **{GAP_CFG.logdir.name}** with **{len(written_files)}** BO-guided seed episodes. "
        "These expert episodes now sit in `train_eps`, so Dreamer will load them into replay before further simulation."
    ))
else:
    print("Set PREPARE_GAP_BRANCH = True to copy the current Dreamer run into a new branch and inject BO-guided expert episodes into replay.")

## Continue The BO-Seeded Branch

Run the next cell only after the replay branch has been prepared. It reuses the current Dreamer checkpoint and extends the episode budget on the BO-seeded branch.

In [ ]:
RUN_GAP_TRAINING = True

if RUN_GAP_TRAINING:
    gap_result = hb.run_dreamer_training(GAP_CFG, run_training=True)
    gap_summary = gap_result.get("after", gap_result)
    display(pd.DataFrame([{key: value for key, value in gap_summary.items() if key != "config_namespace"}]))
else:
    print("Set RUN_GAP_TRAINING = True after the BO-guided seed episodes have been written into the branch replay directory.")

In [ ]:
analysis_logdir = GAP_CFG.logdir if (GAP_CFG.logdir / "metrics.jsonl").exists() else FULL_LOGDIR
analysis_metrics = hb.load_metrics_frame(analysis_logdir)
analysis_train = analysis_metrics.loc[analysis_metrics["train_return"].notna()].copy()
analysis_train = analysis_train.sort_values("step").reset_index(drop=True)
analysis_best = analysis_train.loc[int(analysis_train["train_return"].idxmax())].copy()

comparison_after = pd.DataFrame([
    {
        "analysis_logdir": str(analysis_logdir),
        "best_train_reward": float(analysis_best["train_return"]),
        "reward_gap_to_bo": float(BO_REFERENCE["reward"] - analysis_best["train_return"]),
        "best_step": int(analysis_best["step"]),
        "best_episode": int(analysis_best["train_episode_index"]),
    }
])
display(comparison_after)
hb.plot_training_diagnostics(analysis_metrics)

## Decision Rule

- If the branched run improves the best train reward or reaches the BO reward with lower volatility, keep the BO-seeded continuation path.
- If the branch becomes more stable but still stalls below BO, widen the BO neighborhood seed bank before touching Dreamer internals.
- If the branch still collapses into catastrophic energy-cost spikes, the next issue is probably action parameterization or constraint handling rather than replay initialization.